In [1]:
import re
import pandas as pd
from sqlalchemy import create_engine, inspect

engine = create_engine("sqlite:///../data/full_results2.db")
db = engine.connect()

inspector = inspect(engine)
tables = inspector.get_table_names()

table_name = None
for name in tables:
    if "result" in name.lower() or "full" in name.lower():
        table_name = name
        break
if table_name is None:
    table_name = tables[0]

df = pd.read_sql(f"SELECT * FROM {table_name}", con=engine)

def clean_name(value):
    if pd.isna(value):
        return value
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u00A0", " ")
    text = text.strip()
    return text

if "name" in df.columns:
    df["cleaned_name"] = df["name"].map(clean_name)
elif "full_name" in df.columns:
    df["cleaned_name"] = df["full_name"].map(clean_name)
else:
    df["cleaned_name"] = df.iloc[:, 0].map(clean_name)

total_names = len(df)
total_cleaned_names = df["cleaned_name"].notna().sum()
unique_names = df["cleaned_name"].nunique(dropna=True)
unique_top_countries = df["top_country"].nunique(dropna=True) if "top_country" in df.columns else None
unique_top_country_langs = df["top_country_lang"].dropna().unique() if "top_country_lang" in df.columns else []
unique_top_country_langs_first_entry = unique_top_country_langs[0] if len(unique_top_country_langs) > 0 else None
unique_langdetect_lang = df["langdetect_lang"].nunique(dropna=True) if "langdetect_lang" in df.columns else None

print("table_name:", table_name)
print("total_names:", total_names)
print("total_cleaned_names:", total_cleaned_names)
print("unique_names:", unique_names)
print("unique_top_countries:", unique_top_countries)
print("unique_top_country_langs_first_entry:", unique_top_country_langs_first_entry)
print("unique_langdetect_lang:", unique_langdetect_lang)

table_name: names
total_names: 727352
total_cleaned_names: 727352
unique_names: 727348
unique_top_countries: 105
unique_top_country_langs_first_entry: None
unique_langdetect_lang: 55


In [7]:
df_latin = pd.read_sql(f"SELECT * FROM {table_name} WHERE name = name_latin", con=engine)

if "top_country" in df_latin.columns:
    unique_latin_countries = df_latin["top_country"].dropna().unique().tolist()
else:
    unique_latin_countries = []

print("Countries with latin names", unique_latin_countries)

Countries with latin names ['United States', 'Panama', 'Ethiopia', 'Bangladesh', 'Costa Rica', 'Hungary', 'Slovenia', 'Georgia', 'Portugal', 'Azerbaijan', 'Saudi Arabia', 'Türkiye', 'Lebanon', 'Egypt', 'Cyprus', 'Indonesia', 'Macao', 'Denmark', 'Spain', 'Malaysia', 'Nigeria', 'Maldives', 'India', 'Moldova, Republic of', 'Peru', 'Afghanistan', 'Colombia', 'Lithuania', 'Mexico', 'Poland', 'Puerto Rico', 'United Arab Emirates', 'Croatia', 'Brazil', 'Ecuador', 'Fiji', 'Botswana', 'Serbia', 'Iceland', 'Honduras', 'Ireland', 'Morocco', 'Iran, Islamic Republic of', 'Hong Kong', 'Namibia', 'Malta', 'Italy', 'Jamaica', 'Uruguay', 'Ghana', 'Djibouti', 'Kuwait', 'Canada', 'Taiwan, Province of China', 'Bulgaria', 'Estonia', 'Argentina', 'Singapore', 'Brunei Darussalam', 'France', 'United Kingdom', 'South Africa', 'Germany', 'Burkina Faso', 'Finland', 'Luxembourg', 'Bahrain', 'Tunisia', 'Netherlands', 'Algeria', 'Norway', 'Palestine, State of', 'Syrian Arab Republic', 'Oman', 'Sudan', 'Belgium', 'C